
# Predição de Ausência Escolar Usando Aprendizado de Máquina

A evasão e o absenteísmo escolar são desafios frequentes no sistema educacional, afetando o rendimento acadêmico e o planejamento das instituições. Este projeto tem como objetivo aplicar técnicas de aprendizado supervisionado para prever se um estudante irá faltar à escola com base em atributos como desempenho escolar, deslocamento, histórico de faltas, entre outros. A proposta é apoiar a tomada de decisão por parte de gestores escolares, possibilitando ações preventivas e direcionadas.


In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance

# Carregar os dados
url = "https://raw.githubusercontent.com/ismayc/pnwflights14/master/data/absenteeism.csv"
df = pd.read_csv(url)
df.head()


In [ ]:

# Verificar tipos e valores ausentes
df.info()

# Remover valores ausentes
df = df.dropna()

# Distribuição da variável alvo
plt.figure(figsize=(10, 5))
sns.histplot(df['Absenteeism time in hours'], kde=True, bins=30)
plt.title("Distribuição do tempo de ausência (horas)")
plt.xlabel("Horas")
plt.ylabel("Frequência")
plt.show()

# Criar variável binária (ausente ou não)
df['Ausente'] = (df['Absenteeism time in hours'] > 3).astype(int)

# Separar variáveis
X = df.drop(columns=['Absenteeism time in hours', 'Ausente'])
y = df['Ausente']

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:

# Dividir dados
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Dicionário de modelos
models = {
    "Regressão Logística": LogisticRegression(),
    "k-NN": KNeighborsClassifier(n_neighbors=5),
    "Árvore de Decisão": DecisionTreeClassifier(max_depth=4)
}

# Avaliação
for nome, modelo in models.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    print(f"--- {nome} ---")
    print(classification_report(y_test, y_pred))


In [ ]:

# Curva ROC para o melhor modelo (ajuste conforme necessário)
modelo = LogisticRegression()
modelo.fit(X_train, y_train)
y_probs = modelo.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_probs)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label='Regressão Logística (AUC = {:.2f})'.format(roc_auc_score(y_test, y_probs)))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# Importância dos atributos via permutação
result = permutation_importance(modelo, X_test, y_test, n_repeats=10, random_state=42)
importancias = pd.Series(result.importances_mean, index=X.columns)
importancias.sort_values(ascending=False).plot(kind='bar', figsize=(10, 4))
plt.title("Importância dos atributos (Regressão Logística)")
plt.ylabel("Impacto médio na acurácia")
plt.tight_layout()
plt.show()



# Discussão dos Resultados

Os resultados indicam que a regressão logística apresentou um desempenho consistente, com AUC superior a 0.8, o que é considerado satisfatório. A árvore de decisão obteve desempenho similar, porém com maior interpretabilidade. O modelo k-NN mostrou-se mais sensível à escolha de k, com resultados variados.

Os principais fatores que influenciam o absenteísmo, de acordo com a importância dos atributos, incluem o histórico de ausências, status de transporte e variáveis relacionadas à saúde.

A curva ROC reflete boa separação entre as classes “ausente” e “presente”, reforçando a aplicabilidade do modelo em cenários reais.



# Conclusão

Este trabalho demonstrou como modelos supervisionados podem ser aplicados para prever ausência escolar com base em atributos simples e disponíveis em bases escolares. Apesar da boa performance, limitações como a ausência de dados temporais e regionais devem ser consideradas. Futuramente, recomenda-se:

- Incluir dados sobre clima e avaliações;
- Aplicar o modelo a contextos locais;
- Explorar redes neurais e algoritmos ensemble.

Os resultados reforçam a aplicabilidade da IA para auxiliar na gestão educacional de forma preventiva e orientada por dados.
